In [1]:
import sys
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
import tqdm
from datetime import datetime 

In [2]:
ipath = '../../inputs/MT_/'

## Nodes

In [3]:
pt_nodes_raw = pd.read_excel(ipath + 'MT_68_TC.xlsx', sheet_name='Noeuds')

In [4]:
pt_nodes = pt_nodes_raw.drop(columns = ['tramName', 'tramB']).rename(columns={'Numero de neoud': 'node_id', 'nom': 'name'})
pt_nodes['geometry'] = pt_nodes.apply(lambda x: Point(x['lat'], x['lng']), 1)
pt_nodes = gpd.GeoDataFrame(pt_nodes, geometry='geometry', crs='EPSG:4326')

## Links

In [5]:
pt_network_raw = pd.read_excel(ipath + 'MT_68_TC.xlsx', sheet_name='Tronçons')

In [6]:
pt_network = pt_network_raw.loc[pt_network_raw['IS CONNECTOR ?'] != 'CONNECTOR']
pt_network = pt_network[['Num Noeud Source', 'Num Noeud Destination', 'CAPATI (persons/h)', 'Vitesse']].copy()

pt_network.rename(columns={
    'Num Noeud Source': 'a',
    'Num Noeud Destination': 'b',
    'CAPATI (persons/h)': 'capacity',
    'Vitesse': 'speed'
}, inplace=True)

pt_network = pt_network.merge(pt_nodes[['name', 'node_id', 'geometry']], left_on='a', right_on='node_id')
pt_network = pt_network.merge(pt_nodes[['name', 'node_id', 'geometry']], left_on='b', right_on='node_id', suffixes=['_a', '_b'])

pt_network['geometry'] = pt_network.apply(lambda x: LineString([x['geometry_a'], x['geometry_b']]), 1)
pt_network.drop(columns=['a', 'b', 'node_id_a', 'node_id_b', 'geometry_a', 'geometry_b'], inplace=True)
pt_network = gpd.GeoDataFrame(pt_network, geometry='geometry', crs='EPSG:4326')

In [7]:
pt_nodes.to_file(ipath + 'MT_68_TC_nodes.gpkg')
pt_network.to_file(ipath + 'MT_68_TC_links.gpkg')